# Project 03: Healthcare Risk Analytics & Patient Outcome Prediction

This notebook runs the reproducible synthetic-data workflow end to end. It creates 12,000 synthetic encounters, validates and analyzes the data, trains four models, evaluates calibration and subgroup metrics, and exports only aggregated reports. It is an educational prototype, not a clinical tool.

**Environment:** Python 3.9+ with `pandas`, `numpy`, `scikit-learn`, `scipy`, `matplotlib`, and `seaborn`. From the project root, run the cells in order. The implementation is in the documented, modular `healthcare_risk_pipeline.py` companion so the same workflow can also be run from a terminal.

## 1. Generate deterministic synthetic data

The generator writes `data/raw/patient_records.csv` and `data/raw/facility_lookup.csv`. Re-running it replaces these synthetic files with the same seeded dataset.

In [ ]:
from pathlib import Path
import runpy

project_root = Path.cwd()
if not (project_root / "generate_healthcare_data.py").exists():
    project_root = project_root.parent
generator_path = project_root / "generate_healthcare_data.py"
if not generator_path.exists():
    raise FileNotFoundError("Open this notebook from the project directory or its child directory.")
runpy.run_path(str(generator_path), run_name="__main__")

## 2. Run preprocessing, EDA, model validation, fairness audit, and exports

The pipeline reports duplicate removal and physiologic-range handling, computes descriptive statistics and age-stratified department rates, evaluates stratified 5-fold cross-validation and a held-out test set, then writes plots, metrics, and a formal model card to `data/processed/`. Sex is excluded from model predictors and used only for the subgroup audit. Cohort and fairness cells below configured minimum sizes are suppressed.

In [ ]:
pipeline_path = project_root / "healthcare_risk_pipeline.py"
if not pipeline_path.exists():
    raise FileNotFoundError(f"Pipeline implementation not found: {pipeline_path}")
results = runpy.run_path(str(pipeline_path), run_name="__main__")

## 3. Review exported artifacts

Key deliverables: `agg_cohort_risk.csv`, `agg_subgroup_fairness.csv`, `model_evaluation_metrics.csv`, `logistic_regression_coefficients.csv`, `random_forest_feature_importances.csv`, `model_card.md`, and saved EDA/calibration/confusion-matrix plots. The cohort exports are based on held-out encounters and contain no encounter identifiers. Avoid treating observational patterns or model explanations as causal evidence.